# Federated Unlearning - Memorization-based Evaluation

In [ ]:
dataset_dir = "./dataset"
root_path= './'

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
CUDA_LAUNCH_BLOCKING=1

In [ ]:
from DL_classification_lib import *
from DL_vision_dataset import *
from DL_vision_dataset_sampling import *
from DL_vision_model import *
from General_utils import *
from Image_utils import *

In [ ]:
from Federated_Learning_lib import *
from DL_check_lib import *

from FL_unlearning_fine_tune_by_tracing_gradient_stat import *
from FL_unlearning_utility import *

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import grad
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.utils as vutils
from torchvision import models, datasets, transforms
from torchvision.models import *

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random
import re
import copy
import time
import math
import logging
import os

In [ ]:
import pandas as pd

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Dataset

### Cifar10

In [ ]:
transform = transforms.Compose([
                      transforms.ToTensor(),
                      #transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
                      transforms.Resize([224,224])
                    ])

In [ ]:
Cifar10_train_dataset = Cifar10(root=dataset_dir, train=True, transform=transform)
Cifar10_test_dataset = Cifar10(root=dataset_dir, train=False, transform=transform)
Cifar_dataset_size = len(Cifar10_train_dataset)

In [ ]:
len(Cifar10_train_dataset)

In [ ]:
Cifar10_train_loader = DataLoader(Cifar10_train_dataset, batch_size =128, shuffle=False)
Cifar10_test_loader = DataLoader(Cifar10_test_dataset, batch_size =128, shuffle=False)

### Cifar100

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
])

transform = transforms.Compose([
      transforms.Resize([32,32]),
      transforms.ToTensor(),
      transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
])

In [ ]:
transform_no_augmentation = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
])

In [ ]:
Cifar100_train_dataset = Cifar100(root=dataset_dir, train=True, transform=transform_train)
Cifar100_train_dataset_no_augmentation = Cifar100(root=dataset_dir, train=True, transform=transform_no_augmentation)
Cifar100_test_dataset = Cifar100(root=dataset_dir, train=False, transform=transform)
Cifar100_dataset_size = len(Cifar100_train_dataset)

In [ ]:
len(Cifar100_train_dataset)

In [ ]:
Cifar100_train_loader = DataLoader(Cifar100_train_dataset, batch_size = 32, shuffle=True)
Cifar100_test_loader = DataLoader(Cifar100_test_dataset, batch_size = 32, shuffle=False)

### EMNIST

In [ ]:
transform = transforms.Compose([
                      transforms.ToTensor()
                    ])
target_transform = lambda y: y - 1

In [ ]:
EMNIST_train_dataset = torchvision.datasets.EMNIST(root=dataset_dir+"/EMNIST/", split ="letters", train=True, download=True, transform=transform, target_transform= target_transform)
EMNIST_test_dataset = torchvision.datasets.EMNIST(root=dataset_dir+"/EMNIST/", split = "letters", train=False, download=True, transform=transform, target_transform= target_transform)
EMNIST_dataset_size = len(EMNIST_train_dataset)

In [ ]:
len(EMNIST_train_dataset)

In [ ]:
len(np.unique(EMNIST_train_dataset.targets))

In [ ]:
np.unique(EMNIST_train_dataset.targets)

In [ ]:
EMNIST_train_loader = DataLoader(EMNIST_train_dataset, batch_size =128, shuffle=False)
EMNIST_test_loader = DataLoader(EMNIST_test_dataset, batch_size =128, shuffle=False)

## Model Architecture

### Resnet18

In [ ]:
def get_classification_model_generator_with_Resnet18(type_name, num_classes, **kwargs):
    
    model_dict={
        "Original":OriginalResNet18
    }
    
    def generator():

        model = model_dict[type_name](num_classes = num_classes,**kwargs)
        reset_state_dict = reset_model_state_dict(model)
        model.load_state_dict(reset_state_dict)
        
        return model

    return generator



class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1, group_norm_groups=4, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        #self.bn1 = nn.Identity()
        self.bn1 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        #self.bn2 = nn.Identity()
        self.bn2 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out



#Original Resnet18
class OriginalResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super(OriginalResNet18, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        # self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        # self.bn1 = nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        # self.relu = nn.ReLU(inplace=True)

        #Define the 4 layers as per the structure
        self.layer1 = self._make_layer(64, 64, 2, stride=1, group_norm_groups=4)
        self.layer2 = self._make_layer(64, 128, 2, stride=2, group_norm_groups=8)
        self.layer3 = self._make_layer(128, 256, 2, stride=2, group_norm_groups=16)
        self.layer4 = self._make_layer(256, 512, 2, stride=2, group_norm_groups=32)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_planes, planes, blocks, stride=1, group_norm_groups=4, downsample=None):
        if stride != 1 or in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False),
            )

        layers = []
        layers.append(BasicBlock(in_planes, planes, stride, group_norm_groups, downsample))
        for _ in range(1, blocks):
            layers.append(BasicBlock(planes, planes, group_norm_groups=group_norm_groups))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

### Resnet34

In [ ]:
def get_classification_model_generator_with_Resnet34(type_name, num_classes, **kwargs):

    def generator():
        model = OriginalResNet34(num_classes=num_classes, **kwargs)
        reset_state_dict = reset_model_state_dict(model)
        model.load_state_dict(reset_state_dict)
        return model

    return generator


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, group_norm_groups=4, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class OriginalResNet34(nn.Module):
    def __init__(self, num_classes=100):
        super(OriginalResNet34, self).__init__()
        self.in_planes = 64

        # self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
        # self.bn1 =  nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        # self.relu = nn.ReLU(inplace=True)
        # self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, 64, blocks=3, stride=1, group_norm_groups=4)
        self.layer2 = self._make_layer(64, 128, blocks=4, stride=2, group_norm_groups=8)
        self.layer3 = self._make_layer(128, 256, blocks=6, stride=2, group_norm_groups=16)
        self.layer4 = self._make_layer(256, 512, blocks=3, stride=2, group_norm_groups=32)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_planes, planes, blocks, stride, group_norm_groups):
        downsample = None
        if stride != 1 or in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False),
            )

        layers = [BasicBlock(in_planes, planes, stride, group_norm_groups, downsample)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(planes, planes, group_norm_groups=group_norm_groups))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        # x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


### VGG9

In [ ]:
def get_classification_model_generator_with_VGG9(num_classes, type_name="Original", **kwargs):
    
    model_dict={
        "Original":VGG9_GN
    }
    
    def generator():

        model = model_dict[type_name](num_classes = num_classes,**kwargs)
        reset_state_dict = reset_model_state_dict(model)
        model.load_state_dict(reset_state_dict)
        
        return model

    return generator



class VGG9_GN(nn.Module):
    def __init__(self, num_classes=26, num_groups=8):
        super(VGG9_GN, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: (64, 14, 14)

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: (128, 7, 7)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            #nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

## Lib

In [ ]:
def drop_neuron_by_indices(layer, indices):
    with torch.no_grad():
        weight = layer.weight.data

        weight[indices] = 0.

In [ ]:
def get_initialized_weights(shape, method="normal", device="cuda"):
    import torch.nn.init as init
    """Returns a new tensor with initialized values based on the chosen method."""
    new_tensor = torch.empty(shape).to(device)  # Create an empty tensor of the given shape
    if method == "xavier_uniform":
        init.xavier_uniform_(new_tensor)
    elif method == "xavier_normal":
        init.xavier_normal_(new_tensor)
    elif method == "kaiming_uniform":
        init.kaiming_uniform_(new_tensor, nonlinearity='relu')
    elif method == "kaiming_normal":
        init.kaiming_normal_(new_tensor, nonlinearity='relu')
    elif method == "normal":
        init.normal_(new_tensor, mean=0.0, std=0.1)
    elif method == "uniform":
        init.uniform_(new_tensor, a=-0.1, b=0.1)
    else:
        raise ValueError(f"Unknown initialization method: {method}")
    
    return new_tensor

In [ ]:
def drop_neuron_by_indices(model, indices_by_keys):
    with torch.no_grad():
        state_dict = copy.deepcopy(model.state_dict())

        for key in indices_by_keys:
            weight = state_dict[key]
            #print(weight.shape)
            
            indices = indices_by_keys[key]
            #print(indices.shape)
            #print(weight.shape)
            if indices == None:
                continue

            idx = indices.t()
            #weight[tuple(idx)] = 0.
            #print(weight[tuple(idx)].shape)

            if len(weight.shape) > 1:
                initialized_weights = get_initialized_weights(weight.shape, method="kaiming_uniform", device="cuda")
                weight[tuple(idx)] = initialized_weights[tuple(idx)]
            else:
                weight[tuple(idx)] = get_initialized_weights(weight[tuple(idx)].shape, method="uniform", device="cuda")
        
            state_dict[key] = weight
            
    return state_dict

In [ ]:
def reset_model_state_dict(model):
    with torch.no_grad():
        state_dict = copy.deepcopy(model.state_dict())

        for key in state_dict.keys():
            weight = state_dict[key]

            if len(weight.shape) > 1:
                weight = get_initialized_weights(weight.shape, method="kaiming_uniform", device="cuda")
            else:
                weight = get_initialized_weights(weight.shape, method="uniform", device="cuda")
        
            state_dict[key] = weight
            
    return state_dict

In [ ]:
def measure_memorization_scores(within_model,without_models,dataset):

    loader = DataLoader(dataset, batch_size = 128, shuffle = False)

    baseline_examples_prob = epoch_test_probability(loader, within_model, only_label=True)

    exclude_examples_probs = []

    for without_model in without_models:
    
        exclude_examples_prob = epoch_test_probability(loader, without_model, only_label=True)
        exclude_examples_probs.append(exclude_examples_prob)

    memorization_scores = baseline_examples_prob - torch.mean(torch.stack(exclude_examples_probs), dim=0)

    return memorization_scores

In [ ]:
def parse_log_file(filepath):
    train_acc_list = []
    train_loss_list = []
    test_acc_list = []
    test_loss_list = []
    val_acc_list = []
    val_loss_list = []

    with open(filepath, 'r') as file:
        for line in file:
            if "Train_Acc" in line:
                parts = line.strip().split(',')
                train_acc = float(parts[0].split(':')[1])
                train_loss = float(parts[1].split(':')[1])
                train_acc_list.append(train_acc)
                train_loss_list.append(train_loss)

            elif "Test_Acc" in line:
                parts = line.strip().split(',')
                test_acc = float(parts[0].split(':')[1])
                test_loss = float(parts[1].split(':')[1])
                test_acc_list.append(test_acc)
                test_loss_list.append(test_loss)

            elif "Val_Acc" in line:
                parts = line.strip().split(',')
                val_acc = float(parts[0].split(':')[1])
                val_loss = float(parts[1].split(':')[1])
                val_acc_list.append(val_acc)
                val_loss_list.append(val_loss)

    return {
        'train_acc': train_acc_list,
        'train_loss': train_loss_list,
        'test_acc': test_acc_list,
        'test_loss': test_loss_list,
        'val_acc': val_acc_list,
        'val_loss': val_loss_list
    }

In [ ]:
def plot_fitting_line(base_accuracies, accuracies, bounds, save_name):
    methods = ['Halimi et al.', 'Liu et al.', 'FedRecovery', 'FedAU', 'FedOSD']
    baseline_colors = ['#ffd700', '#ff7f0e', '#8c564b', '#9467bd', '#e377c2'] 
    
    plt.figure(figsize=(8, 6))
    plt.rcParams.update({
        'font.size': 18,         # 字体大小整体调大
        'font.weight': 'bold',   # 字体加粗
        'axes.labelweight': 'bold',  # 坐标轴标题加粗
        'axes.titlesize': 20,        # 坐标轴标题字体大小
        'xtick.labelsize': 18,       # x轴刻度字体大小
        'ytick.labelsize': 18,       # y轴刻度字体大小
        'legend.fontsize': 16        # 图例字体大小
    })

    normal_high_memorization_scores_accuries = base_accuracies[0]
    retraining_high_memorization_scores_accuries = base_accuracies[1]
    fine_tune_high_memorization_scores_accuries = base_accuracies[2]
    
    #Plt
    plt.plot(range(len(bounds)),normal_high_memorization_scores_accuries,color='blue', label="Normal Training")
    plt.plot(range(len(bounds)),retraining_high_memorization_scores_accuries, color='green', label="Retraining")
    #plt.plot(range(len(bounds)),unlearning_high_memorization_scores_accuries, color='orange', label="Unlearning")
    plt.plot(range(len(bounds)),fine_tune_high_memorization_scores_accuries, color='red', label="Ours")
    
    for i, accuracy in enumerate(accuracies):
        plt.plot(range(len(bounds)), accuracy, color=baseline_colors[i], label=methods[i], linestyle='--')
    
    selected_bounds = [5, 20, 35, 50, 65, 80, 95]
    plt.xticks(
        [bounds.index(b) for b in selected_bounds],
        [100 - b for b in selected_bounds]
    )
    plt.ylim(0, 1)
    plt.ylabel("Unlearning Acc.")
    plt.xlabel("Top Memorization Score Examples \n by Percentile (%)")
    
    plt.legend()
    
    plt.savefig("./figures/result/Unlearning/"+save_name, dpi=600)
    
    plt.show()

In [ ]:
def plot_time_line(accuracies, save_name, retraining_epoches=100, epoches = 30):

    test_acc_retraining_0, test_acc_retraining_1, test_acc_retraining_2, test_acc_unlearning = accuracies
    
    plt.figure(figsize=(8, 6))
    plt.rcParams.update({
        'font.size': 18,         # 字体大小整体调大
        'font.weight': 'bold',   # 字体加粗
        'axes.labelweight': 'bold',  # 坐标轴标题加粗
        'axes.titlesize': 20,        # 坐标轴标题字体大小
        'xtick.labelsize': 18,       # x轴刻度字体大小
        'ytick.labelsize': 18,       # y轴刻度字体大小
        'legend.fontsize': 16        # 图例字体大小
    })
    
    # Plot the data
    plt.plot(range(retraining_epoches), test_acc_retraining_0, label='Retraining 0', color='#006400', linestyle='--', linewidth=2)
    plt.plot(range(retraining_epoches), test_acc_retraining_1, label='Retraining 1', color='#228B22', linestyle='--', linewidth=2)
    plt.plot(range(retraining_epoches), test_acc_retraining_2, label='Retraining 2', color='#7CFC00', linestyle='--', linewidth=2)
    plt.plot(range(epoches), test_acc_unlearning, label='Ours',color='red', linewidth=2)
    
    # Beautify the plot
    # plt.title('Epoch Analysis', fontsize=16)
    plt.xlabel('Epoch')
    plt.ylabel('Test Accuracy')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    plt.savefig("./figures/result/Time/" + save_name, dpi=600)
    
    plt.show()

In [ ]:
def calculate_average_value_errors(data_list):
    # stack the arrays along a new axis (axis=0)
    data = np.stack(data_list, axis=0)
    
    # compute the mean along axis=0
    average_value = np.mean(data, axis=0)
    
    # compute the standard deviation along axis=0
    if data.shape[0] == 1:
        std_dev = np.std(data, axis=1)[0]
    else:
        std_dev = np.std(data, axis=1)[0]
    # compute the standard error of the mean
    # standard_error = std_dev / np.sqrt(data.shape[0])
    
    # now you have: average_value ± standard_error
    return average_value, std_dev

In [ ]:
def compute_memorization_accuries_by_model_groups(measurement, models):
    group_accuracy = []
    
    for model in models:
        accuracy = measurement.compute_memorization_accuries_by_groups(model)
        group_accuracy.append(accuracy)
    
    #print(group_accuracy)
    data = np.stack(group_accuracy, axis=0)
    if data.ndim == 2:
        data = data[None, :, :]
    #print(data)
    average_value = np.mean(data, axis=1)[0].tolist()
    #print(average_value)
    std_dev = np.std(data, axis=1)[0].tolist()
    #print(std_dev)
    # compute the standard error of the mean
    # standard_error = std_dev / np.sqrt(data.shape[0])
    
    # now you have: average_value ± standard_error
    return average_value, std_dev

In [ ]:
def compute_memorization_accuries_by_model_groups2(measurement, models):
    group_accuracy = []
    
    for model in models:
        accuracy = measurement.compute_memorization_accuries_by_groups(model)
        group_accuracy.append(accuracy)
    
    #print(group_accuracy)
    data = np.stack(group_accuracy, axis=0)
    if data.ndim == 2:
        data = data[None, :, :]
    #print(data)
    average_value = np.mean(data, axis=0)[0].tolist()
    #print(average_value)
    std_dev = np.std(data, axis=1)[0].tolist()
    #print(std_dev)
    # compute the standard error of the mean
    # standard_error = std_dev / np.sqrt(data.shape[0])
    
    # now you have: average_value ± standard_error
    return average_value, std_dev

In [ ]:
def compute_accuries_on_models(loader, models):
    group_accuracy = []

    for model in models:
        accuracy, _ = epoch_test(loader, model)
        group_accuracy.append(accuracy)

    data = np.array(group_accuracy)

    average_value = np.mean(data)

    std_dev = np.std(data, axis=0)
    
    # compute the standard error of the mean
    # standard_error = std_dev / np.sqrt(data.shape[0])
    
    # now you have: average_value ± standard_error
    return average_value, std_dev

In [ ]:
def read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name", root_path='./'):
    models = []

    if loop_func == "Name":
        for i in range(loop_range):
            loop_name = [str(i) if s == "LOOP_TAG" else s for s in name]
            model_name = make_name(loop_name,"")
            model_path = make_path(root_path,path,model_name)
            model = load_model_based_generator(model_generator, model_path)
        
            models.append(model)

    elif loop_func == "Directory":
        for i in range(loop_range):
            loop_path = path.replace("LOOP_TAG", str(i))
            model_name = make_name(name,"")
            model_path = make_path(root_path,loop_path,model_name)
            model = load_model_based_generator(model_generator, model_path)
        
            models.append(model)

    return models

# Unlearning Performance

## FedAvg IID Cifar100

In [ ]:
target_client_index = 0
client_number = 10

In [ ]:
key_words = ["Cifar100","Resnet34","IID"]
dataset_name = key_words[0]
training_dataset = Cifar100_train_dataset_no_augmentation
test_dataset = Cifar100_test_dataset

In [ ]:
#Data
sampling_name = make_name([*key_words,"Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
target_dataset = SamplerBasedOnIndices(training_dataset,sampling_indices[target_client_index])

In [ ]:
mem_score_name = make_name([*key_words,"MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
#Measurement
bounds = list(range(100,-1,-20))
bounds = [100,95,90,85,80,0]

In [ ]:
measurement = MemorizationMeasurement(target_dataset, memorization_scores, bounds = bounds)

In [ ]:
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
target_loader = DataLoader(target_dataset, batch_size=128, shuffle=False)

In [ ]:
name = [*key_words,"Normal","Retraining","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

retraining_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
retraining_average_value, retraining_standard_error = compute_memorization_accuries_by_model_groups(measurement, retraining_models)
test_retraining_average_value, test_retraining_standard_error = compute_accuries_on_models(test_loader, retraining_models)
target_retraining_average_value, target_retraining_standard_error = compute_accuries_on_models(target_loader, retraining_models)

retraining_res = {
    'average_value': retraining_average_value,
    'standard_error': retraining_standard_error,
    'test_average_value': test_retraining_average_value,
    'test_standard_error': test_retraining_standard_error,
    'target_average_value': target_retraining_average_value,
    'target_standard_error': target_retraining_standard_error
}

In [ ]:
name = [*key_words,"Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

our_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
our_average_value, our_standard_error = compute_memorization_accuries_by_model_groups(measurement, our_models)
test_our_average_value, test_our_standard_error = compute_accuries_on_models(test_loader, our_models)
target_our_average_value, target_our_standard_error = compute_accuries_on_models(target_loader, our_models)

our_res = {
    'average_value': our_average_value,
    'standard_error': our_standard_error,
    'test_average_value': test_our_average_value,
    'test_standard_error': test_our_standard_error,
    'target_average_value': target_our_average_value,
    'target_standard_error': target_our_standard_error
}

In [ ]:
# groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
# models = ['PGA', 'Hessian', 'FedRecovery', 'FedAU', 'FedOSD', 'Ours', 'Retrained Baseline',]

# group_labels = ["PGA","Hessian","FedRecovery","FedAU","FedOSD","our","retraining"]

# results = {
#     'retraining': retraining_res,
#     'our': our_res,
#     'PGA': PGA_res,
#     'Hessian': Hessian_res,
#     'FedRecovery': FedRecovery_res,
#     'FedAU': FedAU_res,
#     'FedOSD': FedOSD_res
# }

groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
models = ['Ours', 'Retrained Baseline',]

group_labels = ["our","retraining"]

results = {
    'retraining': retraining_res,
    'our': our_res,
}

accuracies = []
for group_label in group_labels:
    res = results[group_label]
    accuracies.append(
        [*res['average_value'], res['target_average_value'], res['test_average_value']]
    )

df = pd.DataFrame(accuracies, index=models, columns=groups)
df_percentage = df * 100
df_percentage = df_percentage.round(2)

df_percentage

In [ ]:
errors = []
for group_label in group_labels:
    res = results[group_label]
    errors.append(
        [*res['standard_error'], res['target_standard_error'], res['test_standard_error']]
    )

df = pd.DataFrame(errors, index=models, columns=groups)
df_errors = df * 100
df_errors = df_errors.round(2)

df_errors

In [ ]:
df = df_percentage.copy()

baseline_row = df.loc['Retrained Baseline']

baseline = baseline_row.drop('Test Acc.')
#baseline = baseline[baseline.index].apply(pd.to_numeric, errors='coerce')

diff_df = df.drop('Retrained Baseline').copy()
#diff_df = diff_df[baseline.index].apply(pd.to_numeric, errors='coerce')

for col in baseline.index:
    diff_df[col] = abs(diff_df[col] - baseline[col])

diff_df

In [ ]:
df_final = df_percentage.copy().astype(object)
columns = df_final.columns
    
def print_row(row):
    diffs = diff_df[columns]
    min_diffs = diffs.min()
    
    for col in columns:
        avg_value = df_percentage.loc[row.name, col]
        error_value = df_errors.loc[row.name, col]
        
        if col != 'Test Acc.' and row.name!='Retrained Baseline':
            diff_value = diff_df.loc[row.name, col]
            if diff_value == min_diffs[col]:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{(\\Rb{{{diff_value:.2f}}})}}"
            else:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{({diff_value:.2f})}}"
        else:
            # diff_value = diff_df.loc[row.name, col]
            row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}}"
        
    return row

df_final = df_final.apply(print_row, axis=1)
df_final

## FedAvg Non-IID Cifar100

In [ ]:
target_client_index = 0
client_number = 10

In [ ]:
key_words = ["Cifar100","Resnet34","Non-IID"]
dataset_name = key_words[0]
training_dataset = Cifar100_train_dataset_no_augmentation
test_dataset = Cifar100_test_dataset

In [ ]:
#Data
sampling_name = make_name([*key_words,"Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
target_dataset = SamplerBasedOnIndices(training_dataset,sampling_indices[target_client_index])

In [ ]:
mem_score_name = make_name([*key_words,"MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
#Measurement
bounds = list(range(100,-1,-20))
bounds = [100,95,90,85,80,0]

In [ ]:
measurement = MemorizationMeasurement(target_dataset, memorization_scores, bounds = bounds)

In [ ]:
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
target_loader = DataLoader(target_dataset, batch_size=128, shuffle=False)

In [ ]:
name = [*key_words,"Normal","Retraining","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

retraining_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
retraining_average_value, retraining_standard_error = compute_memorization_accuries_by_model_groups(measurement, retraining_models)
test_retraining_average_value, test_retraining_standard_error = compute_accuries_on_models(test_loader, retraining_models)
target_retraining_average_value, target_retraining_standard_error = compute_accuries_on_models(target_loader, retraining_models)

retraining_res = {
    'average_value': retraining_average_value,
    'standard_error': retraining_standard_error,
    'test_average_value': test_retraining_average_value,
    'test_standard_error': test_retraining_standard_error,
    'target_average_value': target_retraining_average_value,
    'target_standard_error': target_retraining_standard_error
}

In [ ]:
name = [*key_words,"Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

our_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
our_average_value, our_standard_error = compute_memorization_accuries_by_model_groups(measurement, our_models)
test_our_average_value, test_our_standard_error = compute_accuries_on_models(test_loader, our_models)
target_our_average_value, target_our_standard_error = compute_accuries_on_models(target_loader, our_models)

our_res = {
    'average_value': our_average_value,
    'standard_error': our_standard_error,
    'test_average_value': test_our_average_value,
    'test_standard_error': test_our_standard_error,
    'target_average_value': target_our_average_value,
    'target_standard_error': target_our_standard_error
}

In [ ]:
# groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
# models = ['PGA', 'Hessian', 'FedRecovery', 'FedAU', 'FedOSD', 'Ours', 'Retrained Baseline',]

# group_labels = ["PGA","Hessian","FedRecovery","FedAU","FedOSD","our","retraining"]

# results = {
#     'retraining': retraining_res,
#     'our': our_res,
#     'PGA': PGA_res,
#     'Hessian': Hessian_res,
#     'FedRecovery': FedRecovery_res,
#     'FedAU': FedAU_res,
#     'FedOSD': FedOSD_res
# }

groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
models = ['Ours', 'Retrained Baseline',]

group_labels = ["our","retraining"]

results = {
    'retraining': retraining_res,
    'our': our_res,
}

accuracies = []
for group_label in group_labels:
    res = results[group_label]
    accuracies.append(
        [*res['average_value'], res['target_average_value'], res['test_average_value']]
    )

df = pd.DataFrame(accuracies, index=models, columns=groups)
df_percentage = df * 100
df_percentage = df_percentage.round(2)

df_percentage

In [ ]:
errors = []
for group_label in group_labels:
    res = results[group_label]
    errors.append(
        [*res['standard_error'], res['target_standard_error'], res['test_standard_error']]
    )

df = pd.DataFrame(errors, index=models, columns=groups)
df_errors = df * 100
df_errors = df_errors.round(2)

df_errors

In [ ]:
df = df_percentage.copy()

baseline_row = df.loc['Retrained Baseline']

baseline = baseline_row.drop('Test Acc.')
#baseline = baseline[baseline.index].apply(pd.to_numeric, errors='coerce')

diff_df = df.drop('Retrained Baseline').copy()
#diff_df = diff_df[baseline.index].apply(pd.to_numeric, errors='coerce')

for col in baseline.index:
    diff_df[col] = abs(diff_df[col] - baseline[col])

diff_df

In [ ]:
df_final = df_percentage.copy().astype(object)
columns = df_final.columns
    
def print_row(row):
    diffs = diff_df[columns]
    min_diffs = diffs.min()
    
    for col in columns:
        avg_value = df_percentage.loc[row.name, col]
        error_value = df_errors.loc[row.name, col]
        
        if col != 'Test Acc.' and row.name!='Retrained Baseline':
            diff_value = diff_df.loc[row.name, col]
            if diff_value == min_diffs[col]:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{(\\Rb{{{diff_value:.2f}}})}}"
            else:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{({diff_value:.2f})}}"
        else:
            # diff_value = diff_df.loc[row.name, col]
            row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}}"
        
    return row

df_final = df_final.apply(print_row, axis=1)
df_final

## FedAvg IID Cifar10

In [ ]:
target_client_index = 0
client_number = 10

In [ ]:
key_words = ["Cifar10","Resnet18","IID"]
dataset_name = key_words[0]

training_dataset = Cifar10_train_dataset
test_dataset = Cifar10_test_dataset

model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

In [ ]:
#Data
sampling_name = make_name([*key_words,"Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
target_dataset = SamplerBasedOnIndices(training_dataset,sampling_indices[target_client_index])

In [ ]:
mem_score_name = make_name([*key_words,"MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
#Measurement
bounds = list(range(100,-1,-20))
bounds = [100,95,90,85,80,0]

In [ ]:
measurement = MemorizationMeasurement(target_dataset, memorization_scores, bounds = bounds)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
target_loader = DataLoader(target_dataset, batch_size=128, shuffle=False)

In [ ]:
name = [*key_words,"Normal","Retraining","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

retraining_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
retraining_average_value, retraining_standard_error = compute_memorization_accuries_by_model_groups(measurement, retraining_models)
test_retraining_average_value, test_retraining_standard_error = compute_accuries_on_models(test_loader, retraining_models)
target_retraining_average_value, target_retraining_standard_error = compute_accuries_on_models(target_loader, retraining_models)

retraining_res = {
    'average_value': retraining_average_value,
    'standard_error': retraining_standard_error,
    'test_average_value': test_retraining_average_value,
    'test_standard_error': test_retraining_standard_error,
    'target_average_value': target_retraining_average_value,
    'target_standard_error': target_retraining_standard_error
}

In [ ]:
name = [*key_words,"Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

our_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
our_average_value, our_standard_error = compute_memorization_accuries_by_model_groups(measurement, our_models)
test_our_average_value, test_our_standard_error = compute_accuries_on_models(test_loader, our_models)
target_our_average_value, target_our_standard_error = compute_accuries_on_models(target_loader, our_models)

our_res = {
    'average_value': our_average_value,
    'standard_error': our_standard_error,
    'test_average_value': test_our_average_value,
    'test_standard_error': test_our_standard_error,
    'target_average_value': target_our_average_value,
    'target_standard_error': target_our_standard_error
}

In [ ]:
# groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
# models = ['PGA', 'Hessian', 'FedRecovery', 'FedAU', 'FedOSD', 'Ours', 'Retrained Baseline',]

# group_labels = ["PGA","Hessian","FedRecovery","FedAU","FedOSD","our","retraining"]

# results = {
#     'retraining': retraining_res,
#     'our': our_res,
#     'PGA': PGA_res,
#     'Hessian': Hessian_res,
#     'FedRecovery': FedRecovery_res,
#     'FedAU': FedAU_res,
#     'FedOSD': FedOSD_res
# }

groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
models = ['Ours', 'Retrained Baseline',]

group_labels = ["our","retraining"]

results = {
    'retraining': retraining_res,
    'our': our_res,
}

accuracies = []
for group_label in group_labels:
    res = results[group_label]
    accuracies.append(
        [*res['average_value'], res['target_average_value'], res['test_average_value']]
    )

df = pd.DataFrame(accuracies, index=models, columns=groups)
df_percentage = df * 100
df_percentage = df_percentage.round(2)

df_percentage

In [ ]:
errors = []
for group_label in group_labels:
    res = results[group_label]
    errors.append(
        [*res['standard_error'], res['target_standard_error'], res['test_standard_error']]
    )

df = pd.DataFrame(errors, index=models, columns=groups)
df_errors = df * 100
df_errors = df_errors.round(2)

df_errors

In [ ]:
df = df_percentage.copy()

baseline_row = df.loc['Retrained Baseline']

baseline = baseline_row.drop('Test Acc.')
#baseline = baseline[baseline.index].apply(pd.to_numeric, errors='coerce')

diff_df = df.drop('Retrained Baseline').copy()
#diff_df = diff_df[baseline.index].apply(pd.to_numeric, errors='coerce')

for col in baseline.index:
    diff_df[col] = abs(diff_df[col] - baseline[col])

diff_df

In [ ]:
df_final = df_percentage.copy().astype(object)
columns = df_final.columns
    
def print_row(row):
    diffs = diff_df[columns]
    min_diffs = diffs.min()
    
    for col in columns:
        avg_value = df_percentage.loc[row.name, col]
        error_value = df_errors.loc[row.name, col]
        
        if col != 'Test Acc.' and row.name!='Retrained Baseline':
            diff_value = diff_df.loc[row.name, col]
            if diff_value == min_diffs[col]:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{(\\Rb{{{diff_value:.2f}}})}}"
            else:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{({diff_value:.2f})}}"
        else:
            # diff_value = diff_df.loc[row.name, col]
            row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}}"
        
    return row

df_final = df_final.apply(print_row, axis=1)
df_final

## FedAvg Non-IID Cifar10

In [ ]:
target_client_index = 0
client_number = 10

In [ ]:
key_words = ["Cifar10","Resnet18","Non-IID"]
dataset_name = key_words[0]

training_dataset = Cifar10_train_dataset
test_dataset = Cifar10_test_dataset

model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

In [ ]:
#Data
sampling_name = make_name([*key_words,"Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
target_dataset = SamplerBasedOnIndices(training_dataset,sampling_indices[target_client_index])

In [ ]:
mem_score_name = make_name([*key_words,"MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
#Measurement
bounds = list(range(100,-1,-20))
bounds = [100,95,90,85,80,0]

In [ ]:
measurement = MemorizationMeasurement(target_dataset, memorization_scores, bounds = bounds)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
target_loader = DataLoader(target_dataset, batch_size=128, shuffle=False)

In [ ]:
name = [*key_words,"Normal","Retraining","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

retraining_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
retraining_average_value, retraining_standard_error = compute_memorization_accuries_by_model_groups(measurement, retraining_models)
test_retraining_average_value, test_retraining_standard_error = compute_accuries_on_models(test_loader, retraining_models)
target_retraining_average_value, target_retraining_standard_error = compute_accuries_on_models(target_loader, retraining_models)

retraining_res = {
    'average_value': retraining_average_value,
    'standard_error': retraining_standard_error,
    'test_average_value': test_retraining_average_value,
    'test_standard_error': test_retraining_standard_error,
    'target_average_value': target_retraining_average_value,
    'target_standard_error': target_retraining_standard_error
}

In [ ]:
name = [*key_words,"Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

our_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
our_average_value, our_standard_error = compute_memorization_accuries_by_model_groups(measurement, our_models)
test_our_average_value, test_our_standard_error = compute_accuries_on_models(test_loader, our_models)
target_our_average_value, target_our_standard_error = compute_accuries_on_models(target_loader, our_models)

our_res = {
    'average_value': our_average_value,
    'standard_error': our_standard_error,
    'test_average_value': test_our_average_value,
    'test_standard_error': test_our_standard_error,
    'target_average_value': target_our_average_value,
    'target_standard_error': target_our_standard_error
}

In [ ]:
# groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
# models = ['PGA', 'Hessian', 'FedRecovery', 'FedAU', 'FedOSD', 'Ours', 'Retrained Baseline',]

# group_labels = ["PGA","Hessian","FedRecovery","FedAU","FedOSD","our","retraining"]

# results = {
#     'retraining': retraining_res,
#     'our': our_res,
#     'PGA': PGA_res,
#     'Hessian': Hessian_res,
#     'FedRecovery': FedRecovery_res,
#     'FedAU': FedAU_res,
#     'FedOSD': FedOSD_res
# }

groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
models = ['Ours', 'Retrained Baseline',]

group_labels = ["our","retraining"]

results = {
    'retraining': retraining_res,
    'our': our_res,
}

accuracies = []
for group_label in group_labels:
    res = results[group_label]
    accuracies.append(
        [*res['average_value'], res['target_average_value'], res['test_average_value']]
    )

df = pd.DataFrame(accuracies, index=models, columns=groups)
df_percentage = df * 100
df_percentage = df_percentage.round(2)

df_percentage

In [ ]:
errors = []
for group_label in group_labels:
    res = results[group_label]
    errors.append(
        [*res['standard_error'], res['target_standard_error'], res['test_standard_error']]
    )

df = pd.DataFrame(errors, index=models, columns=groups)
df_errors = df * 100
df_errors = df_errors.round(2)

df_errors

In [ ]:
df = df_percentage.copy()

baseline_row = df.loc['Retrained Baseline']

baseline = baseline_row.drop('Test Acc.')
#baseline = baseline[baseline.index].apply(pd.to_numeric, errors='coerce')

diff_df = df.drop('Retrained Baseline').copy()
#diff_df = diff_df[baseline.index].apply(pd.to_numeric, errors='coerce')

for col in baseline.index:
    diff_df[col] = abs(diff_df[col] - baseline[col])

diff_df

In [ ]:
df_final = df_percentage.copy().astype(object)
columns = df_final.columns
    
def print_row(row):
    diffs = diff_df[columns]
    min_diffs = diffs.min()
    
    for col in columns:
        avg_value = df_percentage.loc[row.name, col]
        error_value = df_errors.loc[row.name, col]
        
        if col != 'Test Acc.' and row.name!='Retrained Baseline':
            diff_value = diff_df.loc[row.name, col]
            if diff_value == min_diffs[col]:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{(\\Rb{{{diff_value:.2f}}})}}"
            else:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{({diff_value:.2f})}}"
        else:
            # diff_value = diff_df.loc[row.name, col]
            row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}}"
        
    return row

df_final = df_final.apply(print_row, axis=1)
df_final

## FedAvg IID EMNIST

In [ ]:
target_client_index = 0
client_number = 10

In [ ]:
key_words = ["EMNIST","VGG9","IID"]
dataset_name = key_words[0]
training_dataset = EMNIST_train_dataset
test_dataset = EMNIST_test_dataset

model_generator = get_classification_model_generator_with_VGG9(26)

In [ ]:
#Data
sampling_name = make_name([*key_words,"Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
target_dataset = SamplerBasedOnIndices(training_dataset,sampling_indices[target_client_index])

In [ ]:
mem_score_name = make_name([*key_words,"MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
#Measurement
bounds = list(range(100,-1,-20))
bounds = [100,95,90,85,80,0]

In [ ]:
measurement = MemorizationMeasurement(target_dataset, memorization_scores, bounds = bounds)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
target_loader = DataLoader(target_dataset, batch_size=128, shuffle=False)

In [ ]:
name = [*key_words,"Normal","Retraining","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

retraining_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
retraining_average_value, retraining_standard_error = compute_memorization_accuries_by_model_groups(measurement, retraining_models)
test_retraining_average_value, test_retraining_standard_error = compute_accuries_on_models(test_loader, retraining_models)
target_retraining_average_value, target_retraining_standard_error = compute_accuries_on_models(target_loader, retraining_models)

retraining_res = {
    'average_value': retraining_average_value,
    'standard_error': retraining_standard_error,
    'test_average_value': test_retraining_average_value,
    'test_standard_error': test_retraining_standard_error,
    'target_average_value': target_retraining_average_value,
    'target_standard_error': target_retraining_standard_error
}

In [ ]:
name = [*key_words,"Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

our_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
our_average_value, our_standard_error = compute_memorization_accuries_by_model_groups(measurement, our_models)
test_our_average_value, test_our_standard_error = compute_accuries_on_models(test_loader, our_models)
target_our_average_value, target_our_standard_error = compute_accuries_on_models(target_loader, our_models)

our_res = {
    'average_value': our_average_value,
    'standard_error': our_standard_error,
    'test_average_value': test_our_average_value,
    'test_standard_error': test_our_standard_error,
    'target_average_value': target_our_average_value,
    'target_standard_error': target_our_standard_error
}

In [ ]:
# groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
# models = ['PGA', 'Hessian', 'FedRecovery', 'FedAU', 'FedOSD', 'Ours', 'Retrained Baseline',]

# group_labels = ["PGA","Hessian","FedRecovery","FedAU","FedOSD","our","retraining"]

# results = {
#     'retraining': retraining_res,
#     'our': our_res,
#     'PGA': PGA_res,
#     'Hessian': Hessian_res,
#     'FedRecovery': FedRecovery_res,
#     'FedAU': FedAU_res,
#     'FedOSD': FedOSD_res
# }

groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
models = ['Ours', 'Retrained Baseline',]

group_labels = ["our","retraining"]

results = {
    'retraining': retraining_res,
    'our': our_res,
}

accuracies = []
for group_label in group_labels:
    res = results[group_label]
    accuracies.append(
        [*res['average_value'], res['target_average_value'], res['test_average_value']]
    )

df = pd.DataFrame(accuracies, index=models, columns=groups)
df_percentage = df * 100
df_percentage = df_percentage.round(2)

df_percentage

In [ ]:
errors = []
for group_label in group_labels:
    res = results[group_label]
    errors.append(
        [*res['standard_error'], res['target_standard_error'], res['test_standard_error']]
    )

df = pd.DataFrame(errors, index=models, columns=groups)
df_errors = df * 100
df_errors = df_errors.round(2)

df_errors

In [ ]:
df = df_percentage.copy()

baseline_row = df.loc['Retrained Baseline']

baseline = baseline_row.drop('Test Acc.')
#baseline = baseline[baseline.index].apply(pd.to_numeric, errors='coerce')

diff_df = df.drop('Retrained Baseline').copy()
#diff_df = diff_df[baseline.index].apply(pd.to_numeric, errors='coerce')

for col in baseline.index:
    diff_df[col] = abs(diff_df[col] - baseline[col])

diff_df

In [ ]:
df_final = df_percentage.copy().astype(object)
columns = df_final.columns
    
def print_row(row):
    diffs = diff_df[columns]
    min_diffs = diffs.min()
    
    for col in columns:
        avg_value = df_percentage.loc[row.name, col]
        error_value = df_errors.loc[row.name, col]
        
        if col != 'Test Acc.' and row.name!='Retrained Baseline':
            diff_value = diff_df.loc[row.name, col]
            if diff_value == min_diffs[col]:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{(\\Rb{{{diff_value:.2f}}})}}"
            else:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{({diff_value:.2f})}}"
        else:
            # diff_value = diff_df.loc[row.name, col]
            row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}}"
        
    return row

df_final = df_final.apply(print_row, axis=1)
df_final

## FedAvg Non-IID EMNIST

In [ ]:
target_client_index = 0
client_number = 10

In [ ]:
key_words = ["EMNIST","VGG9","Non-IID"]
dataset_name = key_words[0]
training_dataset = EMNIST_train_dataset
test_dataset = EMNIST_test_dataset

model_generator = get_classification_model_generator_with_VGG9(26)

In [ ]:
#Data
sampling_name = make_name([*key_words,"Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
target_dataset = SamplerBasedOnIndices(training_dataset,sampling_indices[target_client_index])

In [ ]:
mem_score_name = make_name([*key_words,"MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
#Measurement
bounds = list(range(100,-1,-20))
bounds = [100,95,90,85,80,0]

In [ ]:
measurement = MemorizationMeasurement(target_dataset, memorization_scores, bounds = bounds)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
target_loader = DataLoader(target_dataset, batch_size=128, shuffle=False)

In [ ]:
name = [*key_words,"Normal","Retraining","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

retraining_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
retraining_average_value, retraining_standard_error = compute_memorization_accuries_by_model_groups(measurement, retraining_models)
test_retraining_average_value, test_retraining_standard_error = compute_accuries_on_models(test_loader, retraining_models)
target_retraining_average_value, target_retraining_standard_error = compute_accuries_on_models(target_loader, retraining_models)

retraining_res = {
    'average_value': retraining_average_value,
    'standard_error': retraining_standard_error,
    'test_average_value': test_retraining_average_value,
    'test_standard_error': test_retraining_standard_error,
    'target_average_value': target_retraining_average_value,
    'target_standard_error': target_retraining_standard_error
}

In [ ]:
name = [*key_words,"Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model","LOOP_TAG"]
path = "model"

our_models = read_model_group(name, path, model_generator, loop_range = 3, loop_func="Name")
our_average_value, our_standard_error = compute_memorization_accuries_by_model_groups(measurement, our_models)
test_our_average_value, test_our_standard_error = compute_accuries_on_models(test_loader, our_models)
target_our_average_value, target_our_standard_error = compute_accuries_on_models(target_loader, our_models)

our_res = {
    'average_value': our_average_value,
    'standard_error': our_standard_error,
    'test_average_value': test_our_average_value,
    'test_standard_error': test_our_standard_error,
    'target_average_value': target_our_average_value,
    'target_standard_error': target_our_standard_error
}

In [ ]:
# groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
# models = ['PGA', 'Hessian', 'FedRecovery', 'FedAU', 'FedOSD', 'Ours', 'Retrained Baseline',]

# group_labels = ["PGA","Hessian","FedRecovery","FedAU","FedOSD","our","retraining"]

# results = {
#     'retraining': retraining_res,
#     'our': our_res,
#     'PGA': PGA_res,
#     'Hessian': Hessian_res,
#     'FedRecovery': FedRecovery_res,
#     'FedAU': FedAU_res,
#     'FedOSD': FedOSD_res
# }

groups = ['100-95', '95-90', '90-85', '85-80', '80-0', 'Unlearning Acc.', 'Test Acc.']
models = ['Ours', 'Retrained Baseline',]

group_labels = ["our","retraining"]

results = {
    'retraining': retraining_res,
    'our': our_res,
}

accuracies = []
for group_label in group_labels:
    res = results[group_label]
    accuracies.append(
        [*res['average_value'], res['target_average_value'], res['test_average_value']]
    )

df = pd.DataFrame(accuracies, index=models, columns=groups)
df_percentage = df * 100
df_percentage = df_percentage.round(2)

df_percentage

In [ ]:
errors = []
for group_label in group_labels:
    res = results[group_label]
    errors.append(
        [*res['standard_error'], res['target_standard_error'], res['test_standard_error']]
    )

df = pd.DataFrame(errors, index=models, columns=groups)
df_errors = df * 100
df_errors = df_errors.round(2)

df_errors

In [ ]:
df = df_percentage.copy()

baseline_row = df.loc['Retrained Baseline']

baseline = baseline_row.drop('Test Acc.')
#baseline = baseline[baseline.index].apply(pd.to_numeric, errors='coerce')

diff_df = df.drop('Retrained Baseline').copy()
#diff_df = diff_df[baseline.index].apply(pd.to_numeric, errors='coerce')

for col in baseline.index:
    diff_df[col] = abs(diff_df[col] - baseline[col])

diff_df

In [ ]:
df_final = df_percentage.copy().astype(object)
columns = df_final.columns
    
def print_row(row):
    diffs = diff_df[columns]
    min_diffs = diffs.min()
    
    for col in columns:
        avg_value = df_percentage.loc[row.name, col]
        error_value = df_errors.loc[row.name, col]
        
        if col != 'Test Acc.' and row.name!='Retrained Baseline':
            diff_value = diff_df.loc[row.name, col]
            if diff_value == min_diffs[col]:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{(\\Rb{{{diff_value:.2f}}})}}"
            else:
                row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}} \\tb{{({diff_value:.2f})}}"
        else:
            # diff_value = diff_df.loc[row.name, col]
            row[col] = f"{avg_value:.2f} \\ts{{\$\\pm\${error_value:.2f}}}"
        
    return row

df_final = df_final.apply(print_row, axis=1)
df_final